In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Environment & Path Configuration
# Auto-detects Google Colab vs Local Jupyter and sets all path variables.
# Run this cell FIRST before any other cell.
# ─────────────────────────────────────────────────────────────────────────────
import os, sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    import subprocess
    repo_path = Path('/content/amazon-ml-challenge-2026')
    if not repo_path.exists():
        subprocess.run(
            ['git', 'clone',
             'https://github.com/SmithC05/amazon-ml-challenge-2026.git',
             str(repo_path)], check=True)
    os.chdir(repo_path)
    sys.path.insert(0, str(repo_path))
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT   = Path('/content/drive/MyDrive/Amazon ML Challenge 2026')
    REPO_ROOT    = repo_path
    DATASET_ROOT = DRIVE_ROOT / '01_Dataset'
    TRAIN_DIR    = DATASET_ROOT / ' raw' / 'train'
    CACHE_DIR    = DATASET_ROOT / 'processed' / 'm2_cache'
    GT_PATH      = TRAIN_DIR / 'train_ground_truth.tsv'
    OUTPUT_DIR   = Path('/content')
else:
    _nb_dir = Path(globals().get('__vsc_ipynb_file__',
                   globals().get('__file__', ''))).resolve().parent
    REPO_ROOT = _nb_dir.parent if _nb_dir.name == 'notebooks' else _nb_dir
    if not (REPO_ROOT / 'src').exists():
        REPO_ROOT = Path.cwd()
    sys.path.insert(0, str(REPO_ROOT / 'src'))
    sys.path.insert(0, str(REPO_ROOT))
    os.chdir(REPO_ROOT)
    DATASET_ROOT = REPO_ROOT / 'dataset'
    TRAIN_DIR    = DATASET_ROOT / 'raw' / 'train'
    CACHE_DIR    = DATASET_ROOT / 'processed' / 'm2_cache'
    GT_PATH      = TRAIN_DIR / 'train_ground_truth.tsv'
    OUTPUT_DIR   = REPO_ROOT / 'output'

print(f"Environment : {'Google Colab' if IS_COLAB else 'Local Jupyter'}")
print(f"REPO_ROOT   : {REPO_ROOT}")
print(f"DATASET_ROOT: {DATASET_ROOT}")
print(f"TRAIN_DIR   : {TRAIN_DIR}")
print(f"CACHE_DIR   : {CACHE_DIR}")
print(f"GT_PATH     : {GT_PATH}")
print(f"OUTPUT_DIR  : {OUTPUT_DIR}")
print(f"cache exists : {CACHE_DIR.exists()}")
print(f"gt exists    : {GT_PATH.exists()}")


# Notebook 05 — Preprocessing Cache
## `src/cache.py` — One-Time M2 Normalization + Parquet Cache

**Member 2 deliverable — cache layer on top of `src/preprocess.py`**

---

## Why

Every experiment that re-normalizes 300k+ entity records from scratch wastes minutes of wall time.
This notebook builds the Parquet cache once, validates it, and benchmarks the speedup.

After running this notebook:
- M3 training passes `--cache-dir cache` to skip re-normalization
- M4 blocking loads `cache/train_source*.parquet` directly
- Advanced experiments start instantly

---

## 1. Setup

In [ ]:
import sys, time
from pathlib import Path
import pandas as pd
import numpy as np

# Repo root on Colab / local
# REPO_ROOT, sys.path set by env_path_config cell

# Data and cache directories — adjust DATA_DIR to your Drive path
DATA_DIR  = TRAIN_DIR  # set by env_path_config cell above
CACHE_DIR = REPO_ROOT / 'cache'

from cache import build_cache, load_cache, load_all_cache, validate_cache, cache_exists
from preprocess import normalize_name, normalize_address

print('Setup OK')
print(f'DATA_DIR  = {DATA_DIR}')
print(f'CACHE_DIR = {CACHE_DIR}')

## 2. Baseline — Raw Normalization Time (no cache)

In [ ]:
# Measure how long raw normalization takes WITHOUT the cache
t0 = time.perf_counter()

s1_raw = pd.read_csv(DATA_DIR / 'train_source1.tsv', sep='\t')
s2_raw = pd.read_csv(DATA_DIR / 'train_source2.tsv', sep='\t')
s3_raw = pd.read_csv(DATA_DIR / 'train_source3.tsv', sep='\t')

for df in (s1_raw, s2_raw, s3_raw):
    df['business_name_norm']    = df['business_name'].apply(normalize_name)
    df['business_address_norm'] = df['business_address'].apply(normalize_address)

raw_elapsed = time.perf_counter() - t0
total_rows = len(s1_raw) + len(s2_raw) + len(s3_raw)

print(f'Raw normalization time : {raw_elapsed:.2f}s')
print(f'Total rows normalized  : {total_rows:,}')
print(f'Rate                   : {total_rows / raw_elapsed:,.0f} rows/s')

## 3. Build the Cache

In [ ]:
print('Building M2 Parquet cache...')
build_timings = build_cache(
    data_dir=DATA_DIR,
    cache_dir=CACHE_DIR,
    split='train',
    force=False,     # set True to force rebuild
)
print(f"\nCache build total: {build_timings['total']:.2f}s")

## 4. Validate — Cached Values Match Fresh M2 Output

In [ ]:
print('Validating cache integrity (sample 500 rows per source)...')
ok = validate_cache(DATA_DIR, CACHE_DIR, split='train', n_check=500)
assert ok, 'Cache validation FAILED — rebuild with force=True'
print('\nAll validation checks passed. Cache matches M2 normalization output.')

## 5. Load Benchmark — Cache vs Raw

In [ ]:
# Measure cache load time
t0 = time.perf_counter()
s1, s2, s3 = load_all_cache(CACHE_DIR, split='train')
cache_load_elapsed = time.perf_counter() - t0

print(f'Cache load time   : {cache_load_elapsed:.3f}s')
print(f'Raw norm time     : {raw_elapsed:.2f}s')
print(f'Speedup           : {raw_elapsed / cache_load_elapsed:.1f}×')

## 6. Inspect Cached Data

In [ ]:
print('S1 cache schema:')
print(s1.dtypes)
print(f'\nS1 rows: {len(s1):,}  S2: {len(s2):,}  S3: {len(s3):,}')
display(s1.head())

In [ ]:
# Distribution of empty normalized addresses
for name, df in [('S1', s1), ('S2', s2), ('S3', s3)]:
    empty = (df['business_address_norm'] == '').sum()
    print(f'{name}: {empty:,} / {len(df):,} ({100*empty/len(df):.1f}%) empty addresses')

---
## Summary

| Metric | Value |
|---|---|
| Total rows cached | *(see above)* |
| Raw normalization time | *(see above)* |
| Cache build time | *(see above)* |
| Cache load time | *(see above)* |
| Speedup factor | *(see above)* |
| Validation | Passed |

**Next steps:**
- M4: load `cache/train_source*.parquet` for blocking
- M3: pass `--cache-dir cache` to `src/train.py`
- Advanced experiments: use `load_all_cache()` for instant data loading

> Cache files are in `.gitignore` — each teammate runs this notebook once on their machine.